# Stress Prediction V27 — Near-Full CV Retuning
2,850개 학습/150개 검증의 반복 Holdout으로 실제 3,000개 전체 학습 환경에 가까운 검증을 수행합니다.

Tree 분위수·Pair 분위수·혼합 비율을 개발 분할에서 탐색하고 독립 감사 분할에서 재검증합니다.

In [3]:
from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.52
PAIR_WEIGHT = 0.15

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_index = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_index])
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")

In [4]:
from sklearn.model_selection import ShuffleSplit

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
validate_inputs(train, test, submission)

X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

TREE_Q_GRID = (0.48, 0.50, 0.52, 0.54, 0.56)
PAIR_Q_GRID = (0.48, 0.50, 0.52, 0.54, 0.56)
PAIR_W_GRID = (0.05, 0.10, 0.15, 0.20, 0.25)
N_NEAR_FULL_SPLITS = 12
N_DEV_SPLITS = 4
VALID_SIZE = 150
V27_TREES = 1200
PAIR_NAMES = list(combinations(PAIR_COLUMNS, 2))


def tree_prediction_matrix(train_features, valid_features, target, seed=42):
    weighted_train = add_feature_copies(train_features)
    weighted_valid = add_feature_copies(valid_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    valid_matrix = preprocessor.transform(weighted_valid)
    forest = ExtraTreesRegressor(
        n_estimators=V27_TREES,
        min_samples_leaf=1,
        max_features=1,
        random_state=seed,
        n_jobs=-1,
    )
    forest.fit(train_matrix, target)
    return np.column_stack(
        [tree.predict(valid_matrix) for tree in forest.estimators_]
    )


def pair_prediction_matrix(train_features, valid_features, target):
    train_rank, valid_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], valid_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            valid_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    output = []
    for first, second in PAIR_NAMES:
        distance = (
            np.abs(valid_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(valid_rank[second][:, None] - train_rank[second][None, :])
        )
        output.append(target_values[np.argmin(distance, axis=1)])
    return np.column_stack(output)


def aggregate_prediction(tree_matrix, pair_matrix, tree_q, pair_q, pair_w):
    tree_part = np.quantile(tree_matrix, tree_q, axis=1)
    pair_part = np.quantile(pair_matrix, pair_q, axis=1)
    return np.clip(
        np.round((1.0 - pair_w) * tree_part + pair_w * pair_part, 2),
        0.0,
        1.0,
    )


splitter = ShuffleSplit(
    n_splits=N_NEAR_FULL_SPLITS,
    test_size=VALID_SIZE,
    random_state=20260803,
)
split_cache = []

for split_id, (train_idx, valid_idx) in enumerate(splitter.split(X), 1):
    fold_train = X.iloc[train_idx]
    fold_valid = X.iloc[valid_idx]
    fold_target = y.iloc[train_idx]
    tree_matrix = tree_prediction_matrix(
        fold_train, fold_valid, fold_target, seed=4200 + split_id
    )
    pair_matrix = pair_prediction_matrix(fold_train, fold_valid, fold_target)
    split_cache.append({
        "split_id": split_id,
        "valid_idx": valid_idx,
        "target": y.iloc[valid_idx].to_numpy(float),
        "tree_matrix": tree_matrix,
        "pair_matrix": pair_matrix,
    })
    print(f"Near-Full split {split_id}/{N_NEAR_FULL_SPLITS} complete")


candidate_rows = []
for tree_q in TREE_Q_GRID:
    for pair_q in PAIR_Q_GRID:
        for pair_w in PAIR_W_GRID:
            split_scores = []
            baseline_scores = []
            for item in split_cache:
                baseline = aggregate_prediction(
                    item["tree_matrix"], item["pair_matrix"], 0.52, 0.52, 0.15
                )
                candidate = aggregate_prediction(
                    item["tree_matrix"], item["pair_matrix"],
                    tree_q, pair_q, pair_w,
                )
                baseline_scores.append(mean_absolute_error(item["target"], baseline))
                split_scores.append(mean_absolute_error(item["target"], candidate))

            baseline_scores = np.asarray(baseline_scores)
            split_scores = np.asarray(split_scores)
            improvements = baseline_scores - split_scores
            candidate_rows.append({
                "tree_q": tree_q,
                "pair_q": pair_q,
                "pair_w": pair_w,
                "dev_MAE": split_scores[:N_DEV_SPLITS].mean(),
                "dev_improvement": improvements[:N_DEV_SPLITS].mean(),
                "audit_MAE": split_scores[N_DEV_SPLITS:].mean(),
                "audit_mean_improvement": improvements[N_DEV_SPLITS:].mean(),
                "audit_worst_improvement": improvements[N_DEV_SPLITS:].min(),
                "audit_wins": int((improvements[N_DEV_SPLITS:] > 0).sum()),
                "all_MAE": split_scores.mean(),
                "all_improvement": improvements.mean(),
            })

results = pd.DataFrame(candidate_rows)
dev_top = results.sort_values(
    ["dev_MAE", "dev_improvement"], ascending=[True, False]
).head(15)
audited = dev_top.sort_values(
    ["audit_wins", "audit_worst_improvement", "audit_MAE"],
    ascending=[False, False, True],
).reset_index(drop=True)

baseline_row = results[
    np.isclose(results.tree_q, 0.52)
    & np.isclose(results.pair_q, 0.52)
    & np.isclose(results.pair_w, 0.15)
].iloc[0]

print("\n기존 최고 Near-Full 평균 MAE:", f"{baseline_row.all_MAE:.6f}")
display(audited)

eligible = audited[
    (audited.audit_wins >= 6)
    & (audited.audit_mean_improvement >= 0.0003)
    & (audited.audit_worst_improvement >= -0.0005)
    & (audited.all_improvement > 0)
]

if eligible.empty:
    print("V27 rejected: Near-Full 감사 기준을 통과하지 못했습니다.")
    print("제출 파일을 생성하지 않습니다.")
    v27_accepted = False
else:
    selected = eligible.iloc[0]
    v27_accepted = True
    print("V27 accepted")
    print(selected[[
        "tree_q", "pair_q", "pair_w", "all_MAE", "all_improvement",
        "audit_MAE", "audit_mean_improvement", "audit_worst_improvement",
        "audit_wins",
    ]])

    final_tree_matrix = tree_prediction_matrix(X, X_test, y, seed=42)
    final_pair_matrix = pair_prediction_matrix(X, X_test, y)
    final_prediction = aggregate_prediction(
        final_tree_matrix,
        final_pair_matrix,
        float(selected["tree_q"]),
        float(selected["pair_q"]),
        float(selected["pair_w"]),
    )
    final_submission = submission.copy()
    final_submission[TARGET] = final_prediction
    final_path = OUTPUT_DIR / "submit_v27_near_full_cv.csv"
    final_submission.to_csv(final_path, index=False, encoding="utf-8")
    print("Saved:", final_path)
    display(final_submission.head(10))



Near-Full split 1/12 complete
Near-Full split 2/12 complete
Near-Full split 3/12 complete
Near-Full split 4/12 complete
Near-Full split 5/12 complete
Near-Full split 6/12 complete
Near-Full split 7/12 complete
Near-Full split 8/12 complete
Near-Full split 9/12 complete
Near-Full split 10/12 complete
Near-Full split 11/12 complete
Near-Full split 12/12 complete

기존 최고 Near-Full 평균 MAE: 0.125994


,tree_q,pair_q,pair_w,dev_MAE,dev_improvement,audit_MAE,audit_mean_improvement,audit_worst_improvement,audit_wins,all_MAE,all_improvement
0,0.52,0.54,0.15,0.122633,3.469447e-18,0.127542,0.000133,-0.000200,5,0.125906,8.888889e-05
1,0.52,0.48,0.10,0.122433,2.000000e-04,0.127742,-0.000067,-0.000800,5,0.125972,2.222222e-05
2,0.52,0.50,0.15,0.122550,8.333333e-05,0.127592,0.000083,-0.000333,4,0.125911,8.333333e-05
3,0.52,0.56,0.15,0.122583,5.000000e-05,0.127700,-0.000025,-0.000533,4,0.125994,4.625929e-18
4,0.52,0.56,0.10,0.122633,6.938894e-18,0.127750,-0.000075,-0.000600,3,0.126044,-5.000000e-05
5,0.52,0.50,0.10,0.122633,3.469447e-18,0.127750,-0.000075,-0.000600,3,0.126044,-5.000000e-05
6,0.52,0.52,0.10,0.122700,-6.666667e-05,0.127825,-0.000150,-0.000733,3,0.126117,-1.222222e-04
7,0.52,0.48,0.15,0.122583,5.000000e-05,0.127583,0.000092,-0.000733,3,0.125917,7.777778e-05
8,0.52,0.54,0.05,0.122500,1.333333e-04,0.127850,-0.000175,-0.001200,3,0.126067,-7.222222e-05
9,0.52,0.56,0.05,0.122383,2.500000e-04,0.127933,-0.000258,-0.001333,3,0.126083,-8.888889e-05


V27 rejected: Near-Full 감사 기준을 통과하지 못했습니다.
제출 파일을 생성하지 않습니다.
